# Criba de motores TTS — métricas y página de escucha

Se ejecuta **después** de los cuadernos 01–05; mide lo que haya en `outputs/criba/` aunque falte algún
motor. Reutiliza el método de tu `compare_xtts_vs_chatterbox`:

- **WER** con Whisper large-v3 (vía `transformers`) en español: palabras que el audio no dice como debe. Detecta
  alucinaciones, saltos y repeticiones.
- **Similitud** con ECAPA-TDNN entre cada audio y la media de las 3 referencias de su voz, en inglés.
- **UTMOSv2**: calidad percibida estimada, de 1 a 5.

Después monta **`escucha.html`**, la página de escucha a ciegas: las versiones de cada frase aparecen
con letras al azar, y la clave para descifrarlas se guarda aparte, en `clave.json`.

**GPU:** T4. **Tiempo estimado:** unos 40 min. Mismo uso que los demás: celda 1, reiniciar sesión,
y el resto en orden. Lo ya medido no se vuelve a medir.

In [ ]:
# 1. Instalación (después: Entorno de ejecución → Reiniciar sesión)
!pip install -q jiwer speechbrain soundfile librosa pandas
!pip install -q git+https://github.com/sarulab-speech/UTMOSv2.git

In [ ]:
# 2. Preparar
import os, sys, subprocess, json
RAMA = "claude/quirky-dirac-x18c3n"
if os.path.isdir("/content/repo"):
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, "https://github.com/DreiSup/spanishnav.al-podcast", "/content/repo"], check=True)
sys.path.insert(0, "/content/repo/criba")
import comun
comun.montar_drive()
ITEMS, _ = comun.cargar_items()
POR_ID = {it["id"]: it for it in ITEMS}
METRICAS = comun.SALIDA / "metricas"
METRICAS.mkdir(parents=True, exist_ok=True)
VARIANTES = sorted(p.name for p in comun.SALIDA.iterdir() if (p / "registro.jsonl").exists())
AUDIOS = [(v, i) for v in VARIANTES for i in POR_ID if (comun.SALIDA / v / f"{i}.wav").exists()]
print("Variantes:", VARIANTES)
print("Audios a medir:", len(AUDIOS))

In [ ]:
# 3. WER con Whisper large-v3 (vía transformers: faster-whisper necesita CUDA 12 y Colab trae CUDA 13)
import re, jiwer, torch
import numpy as np, soundfile as sf, librosa
from transformers import pipeline

def normaliza(t):
    t = t.lower()
    t = re.sub(r"[^\wáéíóúüñ\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

def audio_16k(ruta):
    a, sr = sf.read(str(ruta), dtype="float32")
    if a.ndim > 1:
        a = a.mean(axis=1)
    return librosa.resample(a, orig_sr=sr, target_sr=16000) if sr != 16000 else a

def cargar_cache(nombre):
    f = METRICAS / nombre
    return {(d["variante"], d["id"]): d for d in map(json.loads, f.read_text(encoding="utf-8").splitlines())} if f.exists() else {}

def anadir(nombre, fila):
    with (METRICAS / nombre).open("a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")

asr_hecho = cargar_cache("asr.jsonl")
asr = pipeline("automatic-speech-recognition", model="openai/whisper-large-v3",
               torch_dtype=torch.float16, device="cuda:0", chunk_length_s=30)
for v, i in AUDIOS:
    if (v, i) in asr_hecho:
        continue
    salida = asr({"raw": audio_16k(comun.SALIDA / v / f"{i}.wav"), "sampling_rate": 16000},
                 generate_kwargs={"language": "spanish", "task": "transcribe"})
    hyp = salida["text"].strip()
    ref = " ".join(POR_ID[i]["frases"])
    fila = {"variante": v, "id": i, "wer": jiwer.wer(normaliza(ref), normaliza(hyp)) if normaliza(hyp) else 1.0,
            "transcripcion": hyp}
    anadir("asr.jsonl", fila)
    asr_hecho[(v, i)] = fila
del asr
comun.liberar()
print("WER hecho:", len(asr_hecho))

In [ ]:
# 4. Similitud de hablante con ECAPA-TDNN
import torch, torchaudio
from speechbrain.inference.speaker import EncoderClassifier
spk = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb", savedir="/content/ecapa",
                                     run_opts={"device": "cuda"})

def embed(ruta):
    sig, sr = torchaudio.load(str(ruta))
    sig = sig.mean(0, keepdim=True)
    if sr != 16000:
        sig = torchaudio.functional.resample(sig, sr, 16000)
    with torch.no_grad():
        e = spk.encode_batch(sig.to("cuda")).squeeze()
    return e / e.norm()

# La identidad se compara siempre con las referencias normales de la voz (los items k también con naval/)
voz_ref = {}
for voz in sorted({it["voz"] for it in ITEMS}):
    clips = sorted((comun.REFS / voz).glob(f"{voz}_ref*.wav"))
    m = torch.stack([embed(c) for c in clips]).mean(0)
    voz_ref[voz] = m / m.norm()
    print(voz, "referencias:", [c.name for c in clips])

sim_hecho = cargar_cache("similitud.jsonl")
for v, i in AUDIOS:
    if (v, i) in sim_hecho:
        continue
    s = float((embed(comun.SALIDA / v / f"{i}.wav") * voz_ref[POR_ID[i]["voz"]]).sum())
    fila = {"variante": v, "id": i, "similitud": s}
    anadir("similitud.jsonl", fila)
    sim_hecho[(v, i)] = fila
del spk
comun.liberar()
print("Similitud hecha:", len(sim_hecho))

In [ ]:
# 5. Calidad estimada con UTMOSv2
import utmosv2
mos_modelo = utmosv2.create_model(pretrained=True)
mos_hecho = cargar_cache("utmos.jsonl")
for v, i in AUDIOS:
    if (v, i) in mos_hecho:
        continue
    fila = {"variante": v, "id": i, "utmos": float(mos_modelo.predict(input_path=str(comun.SALIDA / v / f"{i}.wav")))}
    anadir("utmos.jsonl", fila)
    mos_hecho[(v, i)] = fila
del mos_modelo
comun.liberar()
print("UTMOS hecho:", len(mos_hecho))

In [ ]:
# 6. Tablas
import pandas as pd
filas = []
for v, i in AUDIOS:
    reg = {}
    for l in (comun.SALIDA / v / "registro.jsonl").read_text(encoding="utf-8").splitlines():
        d = json.loads(l)
        if d["id"] == i and d.get("ok"):
            reg = d
    filas.append({"variante": v, "id": i, "voz": POR_ID[i]["voz"], "ref": POR_ID[i]["ref"],
                  "wer": asr_hecho[(v, i)]["wer"], "similitud": sim_hecho[(v, i)]["similitud"],
                  "utmos": mos_hecho[(v, i)]["utmos"], "seg_audio": reg.get("segundos_audio"),
                  "seg_gen": reg.get("segundos_generacion"), "transcripcion": asr_hecho[(v, i)]["transcripcion"]})
df = pd.DataFrame(filas)
df["alucinacion_probable"] = df["wer"] > 0.15
df.to_csv(METRICAS / "detalle.csv", index=False)

resumen = df.groupby("variante").agg(
    wer_medio=("wer", "mean"), items_con_wer_alto=("alucinacion_probable", "sum"),
    similitud=("similitud", "mean"), utmos=("utmos", "mean"),
    rtf=("seg_gen", "sum")).reset_index()
resumen["rtf"] = resumen["rtf"] / df.groupby("variante")["seg_audio"].sum().values
resumen = resumen.sort_values("wer_medio")
resumen.to_csv(METRICAS / "resumen.csv", index=False)
por_voz = df.pivot_table(index="variante", columns="voz", values=["wer", "similitud"], aggfunc="mean")
por_voz.to_csv(METRICAS / "por_voz.csv")
pd.set_option("display.width", 200)
print("=== RESUMEN (WER: menos es mejor; similitud y UTMOS: más es mejor; rtf<1 es más rápido que tiempo real)")
print(resumen.round(3).to_string(index=False))
print("\n=== POR VOZ")
print(por_voz.round(3).to_string())
print("\n=== Peores WER (posibles alucinaciones)")
print(df.sort_values("wer", ascending=False).head(15)[["variante", "id", "wer", "transcripcion"]].to_string(index=False))

In [ ]:
# 7. Página de escucha a ciegas (el código está en criba/escucha.py)
# Elige qué se escucha: 14 items representativos y las variantes que quieras (por defecto, todas).
import escucha
ITEMS_ESCUCHA = ["n01", "n05", "n10", "n14", "n17", "n23", "n25", "v02", "v05", "v09", "b03", "b10", "k04", "k09"]
escucha.construir_pagina(ITEMS_ESCUCHA, VARIANTES, nombre="escucha")

In [ ]:
# 8. Desconectar la GPU
comun.desconectar()